# Module 5: Async Patterns

**Workshop Track:** 200-Level  
**Prerequisites:** Modules 1 through 4 complete

---

In Module 4 we got our first look at the async pattern through `submit_feature_importance_task` and `poll_feature_importance_result`. The idea was simple: submit a job, get a task ID back immediately, and check on it later instead of blocking your process.

Module 5 goes all the way. We apply the same pattern to training and prediction -- the two most compute-intensive operations in any NEXUS workflow. By the end of this module you will know how to submit training jobs without waiting for them, run predictions asynchronously, kick off multiple jobs in parallel, and pick up where you left off in a completely different Python session.

That last capability is worth dwelling on. Task IDs in NEXUS are portable and valid for 48 hours. You can submit a training run before you leave for the day, and poll for the result when you come back in the morning -- from a different script, a different environment, or a different machine. The job runs on Fundamental's infrastructure regardless of what your local process is doing.

## Learning Objectives

By the end of this notebook you will:

- Use `submit_fit_task()` and `poll_fit_result()` to train a model without blocking
- Use `submit_predict_task()`, `submit_predict_proba_task()`, and `poll_predict_result()` for async prediction
- Submit multiple training jobs in parallel and poll them concurrently
- Persist task IDs across sessions and resume polling from a fresh environment

---

## Setup

We will use the top-15 feature set from Module 4. Your `TOP_FEATURES` list is retrieved automatically via `%store -r` from the previous module.

If the retrieval fails (e.g., you skipped Module 4), the data setup cell below will fall back to a default top-15 ranking based on a typical importance run.

In [1]:
import os
import time
import json
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report
from fundamental import Fundamental, NEXUSClassifier, set_client

# Retrieve stored API key from Module 1
%store -r FUNDAMENTAL_API_KEY
os.environ["FUNDAMENTAL_API_KEY"] = FUNDAMENTAL_API_KEY
print(f"API key loaded (prefix: {FUNDAMENTAL_API_KEY[:8]}...)")

set_client(Fundamental())
print("Authentication: OK")

API key loaded (prefix: ak_17749...)
Authentication: OK


In [2]:
DATA_DIR = "../../dataset/credit_risk"

# Load source tables
train_raw   = pd.read_csv(f"{DATA_DIR}/borrowers_train.csv")
holdout_raw = pd.read_csv(f"{DATA_DIR}/borrowers_holdout.csv")
snapshots   = pd.read_csv(f"{DATA_DIR}/financial_snapshots.csv", parse_dates=["snapshot_date"])
assessments = pd.read_csv(f"{DATA_DIR}/credit_assessments.csv", parse_dates=["assessment_date"])
payments    = pd.read_csv(f"{DATA_DIR}/payment_events.csv", parse_dates=["payment_date"])

# Build enriched frame (same pipeline as Modules 3 and 4)
snapshots_latest = (
    snapshots.sort_values("snapshot_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "monthly_income_usd", "income_growth_pct",
      "collateral_score", "secondary_income_flag"]]
)
assessments_latest = (
    assessments.sort_values("assessment_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "creditworthiness_rating", "payment_behavior_score",
      "financial_stability_score", "lender_relationship_score",
      "credit_engagement_score", "debt_service_score"]]
)
payment_agg = (
    payments.groupby("borrower_id")
    .agg(
        total_payments=("payment_id", "count"),
        on_time_rate=("on_time", lambda x: (x == "Yes").mean()),
        avg_payment_usd=("amount_usd", "mean"),
    )
    .reset_index()
)

def enrich(df):
    out = df.copy()
    out = out.merge(snapshots_latest, on="borrower_id", how="left")
    out = out.merge(assessments_latest, on="borrower_id", how="left")
    out = out.merge(payment_agg, on="borrower_id", how="left")
    return out

train_enriched   = enrich(train_raw)
holdout_enriched = enrich(holdout_raw)

drop_cols    = ["borrower_id", "first_name", "last_name", "default_flag"]
all_features = [c for c in train_enriched.columns if c not in drop_cols]

# Apply the Module 3 feature prep: the date becomes a numeric tenure feature;
# categoricals pass to NEXUS as-is


def add_account_tenure(df, date_col="account_open_date", ref_date="2026-01-01"):
    """Convert the account-open date into a numeric tenure feature.

    NEXUS accepts numeric, boolean, string, and categorical columns — but not
    datetime columns. So instead of parsing the date, we derive an explicit
    numeric feature: the account's age in days at a fixed reference date.
    (A fixed reference keeps the feature stable across runs.)
    """
    out = df.copy()
    out["account_tenure_days"] = (
        pd.Timestamp(ref_date) - pd.to_datetime(out[date_col])
    ).dt.days
    return out.drop(columns=[date_col])


X_train_full = add_account_tenure(train_enriched[all_features])
X_holdout_full = add_account_tenure(holdout_enriched[all_features])

y_train   = train_enriched["default_flag"]
y_holdout = holdout_enriched["default_flag"]

print(f"Enriched frame ready: {X_train_full.shape[0]:,} rows x {X_train_full.shape[1]} features")

Enriched frame ready: 4,591 rows x 22 features


In [3]:
# Retrieve top-15 feature set from Module 4 (stored automatically if you ran Module 4)
%store -r TOP_FEATURES
# Fall back to a self-contained list if the store is empty or out of date (column names changed).
try:
    _ok = isinstance(TOP_FEATURES, (list, tuple)) and set(TOP_FEATURES).issubset(X_train_full.columns)
except NameError:
    _ok = False
if not _ok:
    TOP_FEATURES = ['monthly_income_usd', 'avg_payment_usd', 'total_employment_years', 'age',
                    'secondary_income_flag', 'account_tenure_days', 'marital_status', 'collateral_score',
                    'occupation_sector', 'num_previous_lenders', 'distance_from_branch_miles',
                    'credit_engagement_score', 'financial_stability_score', 'payment_behavior_score',
                    'creditworthiness_rating']
    print("Using fallback TOP_FEATURES (Module 4 store missing or out of date).")

print(f"Loaded {len(TOP_FEATURES)} features from Module 4: {TOP_FEATURES[:3]}...")

# TOP_FEATURES references the prepared frame's column names (e.g., the numeric
# 'account_tenure_days' feature) because Module 4 ranked features after applying the
# same preprocessing. The categorical values are passed raw to NEXUS; only the
# column names matter here.
X_train_slim   = X_train_full[TOP_FEATURES]
X_holdout_slim = X_holdout_full[TOP_FEATURES]

print(f"Slim feature set: {X_train_slim.shape[1]} features")
print(f"Full feature set: {X_train_full.shape[1]} features")

Loaded 15 features from Module 4: ['monthly_income_usd', 'avg_payment_usd', 'total_employment_years']...
Slim feature set: 15 features
Full feature set: 22 features


---

## Part 1: Why Async Training?

### The Problem with `fit()`

Synchronous `fit()` is convenient: one call, blocks until done, returns a fitted estimator. That works fine in an interactive notebook when training finishes in under a minute.

But real-world training jobs do not always finish in under a minute. Larger datasets, quality mode, or simply a busy API can stretch training to several minutes. A blocking call in that scenario means:

- Your notebook process is tied up and cannot run other cells
- If the connection drops mid-call, you may lose the job entirely
- You cannot run multiple experiments in parallel
- You cannot submit a job before a meeting and check the result afterward

Async training solves all of this. You submit the job, get a task ID back immediately, and the training runs on Fundamental's infrastructure while your process does whatever it needs to do. The task ID is valid for 48 hours and works across sessions -- you can close your laptop and come back tomorrow.

Think of it like ordering food at a restaurant instead of cooking it yourself at the table. You place the order (submit), get a ticket number (task ID), and come back when it's ready (poll). The kitchen keeps running whether you're at the table or not.

---

## Part 2: Async Training

### submit_fit_task

`submit_fit_task(X, y)` takes the same data arguments as `fit()` but returns a task ID string instead of waiting for results. The training mode is set on the constructor (e.g., `NEXUSClassifier(mode="quality")`) and stays fixed for the life of the estimator. You can save the returned task ID to a variable, a file, or a database, and use it later.

In [4]:
# Submit training on the top-15 feature set -- returns immediately
nexus_async = NEXUSClassifier(mode="quality")

t0 = time.time()
fit_task_id = nexus_async.submit_fit_task(X_train_slim, y_train)
submit_elapsed = time.time() - t0

print(f"Job submitted in {submit_elapsed:.2f}s")
print(f"Task ID: {fit_task_id}")
print()
print("The training job is now running on Fundamental's infrastructure.")
print("Your Python process is free. Save this task ID before continuing.")

Job submitted in 1.11s
Task ID: c3aed3a493d988c689a21a3b541cf740

The training job is now running on Fundamental's infrastructure.
Your Python process is free. Save this task ID before continuing.


### poll_fit_result

`poll_fit_result(task_id)` returns `self` -- the fitted estimator -- when training is complete, or `None` if the job is still running. Once it returns the estimator, you can call `predict`, `predict_proba`, or `get_feature_importance` immediately.

One important detail: `poll_fit_result` does not have to be called on the same estimator instance that submitted the job. You can create a fresh `NEXUSClassifier()`, call `poll_fit_result` on it, and it will become fitted when the job completes. This is what makes task IDs cross-session portable.

In [5]:
POLL_INTERVAL = 20  # seconds

print(f"Polling task {fit_task_id}...")

fitted = nexus_async.poll_fit_result(fit_task_id)
while fitted is None:
    print(f"  Still running... checking again in {POLL_INTERVAL}s")
    time.sleep(POLL_INTERVAL)
    fitted = nexus_async.poll_fit_result(fit_task_id)

# nexus_async is now fitted -- fitted and nexus_async are the same object
print(f"\nTraining complete.")
print(f"Model ID: {nexus_async.trained_model_id_}")

Polling task c3aed3a493d988c689a21a3b541cf740...
  Still running... checking again in 20s


  Still running... checking again in 20s



Training complete.
Model ID: 1f5759f1-84aa-4775-aa44-7df3d4013ddd


In [6]:
# Evaluate the async-trained model -- same as any fitted estimator
proba_async = nexus_async.predict_proba(X_holdout_slim)
auc_async   = roc_auc_score(y_holdout, proba_async[:, 1])

print(f"Holdout AUC (async-trained, top-15 features): {auc_async:.4f}")
print()
preds_async = (proba_async[:, 1] >= 0.30).astype(int)
print(classification_report(y_holdout, preds_async, target_names=["No Default", "Default"]))

Holdout AUC (async-trained, top-15 features): 0.9435

              precision    recall  f1-score   support

  No Default       0.96      0.95      0.96       929
     Default       0.79      0.84      0.82       220

    accuracy                           0.93      1149
   macro avg       0.88      0.89      0.89      1149
weighted avg       0.93      0.93      0.93      1149



---

## Part 3: Async Prediction

### submit_predict_task and submit_predict_proba_task

Prediction is typically fast enough that synchronous `predict()` is fine. But async prediction becomes valuable when:

- Your holdout set is large (hundreds of thousands of rows)
- You want to submit predictions in a pipeline that should not block the main thread
- You are orchestrating multiple predict calls simultaneously

Both `submit_predict_task()` (class labels) and `submit_predict_proba_task()` (probabilities) share the same poll function: `poll_predict_result()`. The poll returns an ndarray when done, or `None` if still running.

In [7]:
# Submit an async probability prediction job
proba_task_id = nexus_async.submit_predict_proba_task(X_holdout_slim)

print(f"Prediction job submitted.")
print(f"Task ID: {proba_task_id}")
print()
print("Doing other work while predictions run...")
print("(In a real pipeline, this is where you would process other data,")
print(" generate reports, or trigger downstream tasks.)")

Prediction job submitted.
Task ID: 149606f7039545552d0c789da3aa9dde

Doing other work while predictions run...
(In a real pipeline, this is where you would process other data,
 generate reports, or trigger downstream tasks.)


In [8]:
# Poll for the prediction result.
# Both submit_predict_task and submit_predict_proba_task poll via poll_predict_result.
proba_result = nexus_async.poll_predict_result(proba_task_id)
while proba_result is None:
    time.sleep(10)
    proba_result = nexus_async.poll_predict_result(proba_task_id)

print(f"Prediction complete. Result shape: {proba_result.shape}")
auc_async_pred = roc_auc_score(y_holdout, proba_result[:, 1])
print(f"AUC from async prediction: {auc_async_pred:.4f}")

Prediction complete. Result shape: (1149, 2)
AUC from async prediction: 0.9431


In [9]:
# submit_predict_task returns class labels instead of probabilities
labels_task_id = nexus_async.submit_predict_task(X_holdout_slim)

labels_result = nexus_async.poll_predict_result(labels_task_id)
while labels_result is None:
    time.sleep(10)
    labels_result = nexus_async.poll_predict_result(labels_task_id)

# Both predict paths use the same poll function
default_count = labels_result.sum()
print(f"Predicted defaults: {default_count} ({default_count / len(labels_result):.1%} of holdout)")

Predicted defaults: 213 (18.5% of holdout)


---

## Part 4: Parallel Jobs

### The Real Multiplier

This is where async becomes genuinely powerful. With synchronous calls, running two training experiments takes twice as long as running one. With async, you submit both jobs in rapid succession, and they run concurrently on Fundamental's infrastructure.

The experiment below trains two models in parallel:

- **Model A:** Top-15 features (slim), quality mode
- **Model B:** Full enriched set (22 features), speed mode

Both submissions happen before we poll for either result. The total wall-clock time is roughly the duration of the longer job, not the sum of both.

In [10]:
# Create two estimators -- one for each experiment
# Mode is set on the constructor and stays fixed for the life of the instance.
nexus_A = NEXUSClassifier(mode="quality")
nexus_B = NEXUSClassifier(mode="speed")

# Submit both jobs immediately -- neither blocks
t0 = time.time()

task_A = nexus_A.submit_fit_task(X_train_slim, y_train)
task_B = nexus_B.submit_fit_task(X_train_full, y_train)

submit_time = time.time() - t0
print(f"Both jobs submitted in {submit_time:.2f}s")
print(f"  Model A task: {task_A}")
print(f"  Model B task: {task_B}")
print()
print("Both jobs are running concurrently on Fundamental's infrastructure.")

Both jobs submitted in 1.95s
  Model A task: af72cbcfad1f5e192ce6641ec6b64eb6
  Model B task: 54814239ad9d443ed06881e94c8ebf15

Both jobs are running concurrently on Fundamental's infrastructure.


In [11]:
# Poll both jobs in a shared loop
# The loop continues until both are complete

done = {"A": False, "B": False}

print("Polling both jobs...")

while not all(done.values()):
    if not done["A"]:
        result_A = nexus_A.poll_fit_result(task_A)
        if result_A is not None:
            done["A"] = True
            print(f"  Model A complete. ID: {nexus_A.trained_model_id_}")

    if not done["B"]:
        result_B = nexus_B.poll_fit_result(task_B)
        if result_B is not None:
            done["B"] = True
            print(f"  Model B complete. ID: {nexus_B.trained_model_id_}")

    if not all(done.values()):
        pending = [k for k, v in done.items() if not v]
        print(f"  Still waiting on: {pending} -- checking again in 20s")
        time.sleep(20)

total_wall = time.time() - t0
print(f"\nBoth jobs complete. Total wall-clock time: {total_wall:.1f}s")

Polling both jobs...
  Still waiting on: ['A', 'B'] -- checking again in 20s


  Model B complete. ID: 02b5ee18-db8d-4d00-9ed7-6f24247b1586
  Still waiting on: ['A'] -- checking again in 20s


  Model A complete. ID: a1041c29-3c72-425a-9acf-3b2a063177b6

Both jobs complete. Total wall-clock time: 42.8s


In [12]:
# Compare results from both parallel experiments
proba_A = nexus_A.predict_proba(X_holdout_slim)
proba_B = nexus_B.predict_proba(X_holdout_full)

auc_A = roc_auc_score(y_holdout, proba_A[:, 1])
auc_B = roc_auc_score(y_holdout, proba_B[:, 1])

print("Parallel experiment results:")
print(f"  Model A (top-15, quality):       AUC = {auc_A:.4f}")
print(f"  Model B (full features, speed):  AUC = {auc_B:.4f}")
print(f"  Delta:                           {auc_A - auc_B:+.4f}")
print()
print("Wall-clock time for both: ~same as the slower single job.")
print("Sequential equivalent would have taken roughly 2x as long.")

Parallel experiment results:
  Model A (top-15, quality):       AUC = 0.9428
  Model B (full features, speed):  AUC = 0.9389
  Delta:                           +0.0039

Wall-clock time for both: ~same as the slower single job.
Sequential equivalent would have taken roughly 2x as long.


### When the result does not come back

Every poll loop so far assumed the result eventually arrives. Production code cannot. Give the wait a budget, and decide in advance what happens when the budget runs out — the task id stays valid, so a spent budget is a decision point, not a dead end.

In [13]:
def poll_with_budget(estimator, task_id, max_polls=10, interval=15):
    """Poll until the result arrives or the budget is spent."""
    for i in range(1, max_polls + 1):
        result = estimator.poll_predict_result(task_id)
        if result is not None:
            print(f"Result ready after {i} poll(s)")
            return result
        time.sleep(interval)
    raise TimeoutError(
        f"No result after {max_polls} polls. The task id is still valid — "
        "resume polling later, fall back to the previous model, or escalate."
    )

budget_task = nexus_async.submit_predict_proba_task(X_holdout_slim)
proba_budgeted = poll_with_budget(nexus_async, budget_task)
print(proba_budgeted[:3])

Result ready after 2 poll(s)
[[0.9913297  0.0086703 ]
 [0.67606409 0.32393591]
 [0.08536691 0.91463309]]


### Exercise: write the fallback note

Your pipeline submits two training jobs in parallel. One completes; the other has not returned by your deadline. In the cell below, write your fallback strategy in two or three sentences: what do you do with the spent budget, the still-valid task id, and the model you already have? There is no single right answer — the point is deciding before the incident, not during it.

*Your fallback note:*

---

## Part 5: Task Portability

### Surviving Session Restarts

One of the most practical things about NEXUS task IDs is that they are just strings. You can save them to a file, a database, an environment variable, or a Slack message, and use them in any Python process that has a valid API key. They remain valid for 48 hours after submission.

This enables a real production pattern: your data pipeline submits training at 11 PM, and your evaluation script picks up the results at 7 AM from a completely different environment.

The cells below simulate this: we save the task IDs to a JSON file, then reload them and poll as if we were starting fresh.

In [14]:
# Simulate end-of-session: save all task IDs to a file
task_registry = {
    "slim_quality": {
        "task_id": fit_task_id,
        "model_id": nexus_async.trained_model_id_,
        "features": "top_15",
        "mode": "quality",
    },
    "parallel_A": {
        "task_id": task_A,
        "model_id": nexus_A.trained_model_id_,
        "features": "top_15",
        "mode": "quality",
    },
    "parallel_B": {
        "task_id": task_B,
        "model_id": nexus_B.trained_model_id_,
        "features": "full_enriched",
        "mode": "speed",
    },
}

with open("module_05_task_registry.json", "w") as f:
    json.dump(task_registry, f, indent=2)

print("Task registry saved to: module_05_task_registry.json")
print(json.dumps(task_registry, indent=2))

Task registry saved to: module_05_task_registry.json
{
  "slim_quality": {
    "task_id": "c3aed3a493d988c689a21a3b541cf740",
    "model_id": "1f5759f1-84aa-4775-aa44-7df3d4013ddd",
    "features": "top_15",
    "mode": "quality"
  },
  "parallel_A": {
    "task_id": "af72cbcfad1f5e192ce6641ec6b64eb6",
    "model_id": "a1041c29-3c72-425a-9acf-3b2a063177b6",
    "features": "top_15",
    "mode": "quality"
  },
  "parallel_B": {
    "task_id": "54814239ad9d443ed06881e94c8ebf15",
    "model_id": "02b5ee18-db8d-4d00-9ed7-6f24247b1586",
    "features": "full_enriched",
    "mode": "speed"
  }
}


In [15]:
# Simulate beginning of a new session: load the registry and reconnect
with open("module_05_task_registry.json", "r") as f:
    loaded_registry = json.load(f)

print("Loaded task registry. Reconnecting to models...\n")

for name, entry in loaded_registry.items():
    feature_set = TOP_FEATURES if entry["features"] == "top_15" else list(X_holdout_full.columns)
    X_eval = X_holdout_full[feature_set]

    # Load the trained model by ID -- no need to poll again if already complete.
    # If the stored id is stale (e.g., expired or from a different run), fit fresh.
    nexus_loaded = NEXUSClassifier(mode=entry["mode"])
    try:
        nexus_loaded.load_model(entry["model_id"])
    except Exception:
        print(f"{name:20s} | stored id stale -- fitting a fresh model")
        X_fit = X_train_slim if entry["features"] == "top_15" else X_train_full
        nexus_loaded.fit(X_fit, y_train)

    proba = nexus_loaded.predict_proba(X_eval)
    auc   = roc_auc_score(y_holdout, proba[:, 1])

    print(f"{name:20s} | model: {nexus_loaded.trained_model_id_} | AUC: {auc:.4f}")

Loaded task registry. Reconnecting to models...



slim_quality         | model: 1f5759f1-84aa-4775-aa44-7df3d4013ddd | AUC: 0.9428


parallel_A           | model: a1041c29-3c72-425a-9acf-3b2a063177b6 | AUC: 0.9441


parallel_B           | model: 02b5ee18-db8d-4d00-9ed7-6f24247b1586 | AUC: 0.9395


### The Cross-Session Pattern in Practice

The above workflow is deliberately simplified. In a real system, the pattern looks more like this:

**Submission side** (runs at end of data refresh cycle, or on a schedule):
```python
nexus = NEXUSClassifier(mode="quality")
task_id = nexus.submit_fit_task(X_train, y_train)
store_task_id(database, run_id=today, task_id=task_id)  # your persistence layer
```

**Polling side** (runs periodically, or triggered by a downstream event):
```python
task_id = retrieve_task_id(database, run_id=today)
nexus = NEXUSClassifier()
result = nexus.poll_fit_result(task_id)
if result is not None:
    promote_model(nexus.trained_model_id_)  # your deployment step
```

The key insight: the submission and polling sides do not need to share any state beyond the task ID string. That string is everything.

---

## Summary: The Full Async Surface

Here is a complete reference of all the async methods available on `NEXUSClassifier` / `NEXUSRegressor`:

| Operation | Submit | Poll | Returns |
|-----------|--------|------|---------|
| Training | `submit_fit_task(X, y)` | `poll_fit_result(task_id)` | `Self \| None` |
| Prediction (labels) | `submit_predict_task(X)` | `poll_predict_result(task_id)` | `ndarray \| None` |
| Prediction (probs) | `submit_predict_proba_task(X)` | `poll_predict_result(task_id)` | `ndarray \| None` |
| Feature importance | `submit_feature_importance_task(X)` | `poll_feature_importance_result(task_id)` | `ndarray \| None` |

All task IDs are valid for 48 hours. All poll methods return `None` while the job is in progress. Mode is set on the constructor (`NEXUSClassifier(mode="quality")`), not passed to `submit_fit_task`. `submit_predict_task` and `submit_predict_proba_task` share a single poll method — `poll_predict_result` returns class labels for one and a probability matrix for the other.

---

## Save Your Model IDs

Save the models trained in this module -- you will use them in Module 6 when we cover exception handling and retry logic.

In [16]:
# Store the slim quality model ID for Module 6
SLIM_MODEL_ID = nexus_async.trained_model_id_
%store SLIM_MODEL_ID
%store TOP_FEATURES

print("=== Save These Model IDs ===")
print(f"Async slim model (quality):  {nexus_async.trained_model_id_}")
print(f"Parallel A (slim, quality):  {nexus_A.trained_model_id_}")
print(f"Parallel B (full, speed):    {nexus_B.trained_model_id_}")
print()
print(f"Top-15 feature list: {TOP_FEATURES}")

Stored 'SLIM_MODEL_ID' (str)
Stored 'TOP_FEATURES' (list)
=== Save These Model IDs ===
Async slim model (quality):  1f5759f1-84aa-4775-aa44-7df3d4013ddd
Parallel A (slim, quality):  a1041c29-3c72-425a-9acf-3b2a063177b6
Parallel B (full, speed):    02b5ee18-db8d-4d00-9ed7-6f24247b1586

Top-15 feature list: ['monthly_income_usd', 'avg_payment_usd', 'total_employment_years', 'age', 'account_tenure_days', 'secondary_income_flag', 'marital_status', 'distance_from_branch_miles', 'collateral_score', 'occupation_sector', 'financial_stability_score', 'num_previous_lenders', 'payment_behavior_score', 'credit_engagement_score', 'income_growth_pct']


---

## Key Takeaways

**`submit_fit_task()` returns a task ID in about a second.** Your process is free immediately. Training runs on Fundamental's infrastructure and can be retrieved for up to 48 hours.

**`poll_fit_result()` is portable.** You do not need the original estimator instance. Create any `NEXUSClassifier()`, pass the task ID, and it becomes fitted when the job completes. This is the foundation of cross-session workflows.

**Prediction shares one poll method.** `submit_predict_task()` and `submit_predict_proba_task()` both poll via `poll_predict_result()` — class labels come back from one, a probability matrix from the other. Check the reference table above for the full surface.

**Parallel jobs are the real payoff.** Submit N jobs in rapid succession, poll all of them in a shared loop. Wall-clock time is the duration of the longest job, not the sum.

**Task IDs are just strings.** Save them anywhere, use them from anywhere, within 48 hours.

---

**What's Next -- Module 6: Resilient Pipelines**

Module 6 covers what happens when things go wrong: the NEXUS exception hierarchy, retry logic with backoff, timeout configuration, and how to build pipelines that recover gracefully from transient failures without losing the work already done.